# Báo cáo Kỹ thuật: Phân tích và Kiểm thử Ma trận (Part 1)

Notebook này trình diễn các thuật toán lõi của Part 1 bao gồm Khử Gauss, tính Định thức, Nghịch đảo và Hạng ma trận. 
Mục tiêu cốt lõi:
1. Xác minh tính đúng đắn bằng phương pháp **Assertion-based Testing**.
2. Phân tích tính ổn định số học qua **Ill-conditioned Matrices**.

In [1]:
from pathlib import Path
import numpy as np
import math

# Tự động cập nhật thay đổi từ các file .py
%load_ext autoreload
%autoreload 2

# Import các module tự viết
from gaussian import gaussian_eliminate, back_substitution
from determinant import determinant
from inverse import inverse
from rank_basis import rank_and_basis

print("✅ Môi trường đã sẵn sàng.")

✅ Môi trường đã sẵn sàng.


## 1. Cơ chế Kiểm chứng (Verification Logic)

Chúng ta sử dụng hàm `verify_system` để kiểm tra nghiệm $x$ của hệ $Ax = b$. 
Sai số được tính bằng **Relative Residual**:
$$\text{Error} = \frac{\|Ax - b\|}{\|b\|}$$
Sử dụng câu lệnh `assert` để đảm bảo sai số không vượt quá $10^{-5}$. Nếu vượt quá, hệ thống sẽ dừng thực thi.

In [2]:
def verify_system(A, x, b, label="Case"):
    A_np = np.array(A, dtype=float)
    x_np = np.array(x, dtype=float)
    b_np = np.array(b, dtype=float)
    
    # Tính sai số phần dư
    residual = np.dot(A_np, x_np) - b_np
    error = np.linalg.norm(residual)
    b_norm = np.linalg.norm(b_np)
    rel_error = error / b_norm if b_norm > 1e-15 else error
    
    # Assertion: Dừng notebook nếu sai số quá lớn
    assert rel_error < 1e-5, f"❌ {label} FAILED: Sai số {rel_error:.2e} vượt ngưỡng!"
    
    print(f"   [OK] {label}: Verified (Relative Residual: {rel_error:.2e})")

## 2. Giải hệ phương trình (Gaussian Elimination)

Thuật toán sử dụng phương pháp **Partial Pivoting** để tìm phần tử trụ lớn nhất, giúp giảm thiểu sai số làm tròn. Chúng ta sẽ thử nghiệm trên các cấu hình ma trận khác nhau (Vuông, Chữ nhật, Suy biến).

In [3]:
print("--- Testing Gaussian Suite ---")
test_cases = [
    {"name": "Regular 3x3", "A": [[2, 1, -1], [-3, -1, 2], [-2, 1, 2]], "b": [8, -11, -3]},
    {"name": "Identity 2x2", "A": [[1, 0], [0, 1]], "b": [5, 10]},
    {"name": "Singular 2x2", "A": [[1, 2], [2, 4]], "b": [5, 10]}
]

for case in test_cases:
    M, x, swaps = gaussian_eliminate(case["A"], case["b"])
    verify_system(case["A"], x, case["b"], label=case["name"])

--- Testing Gaussian Suite ---
   [OK] Regular 3x3: Verified (Relative Residual: 6.38e-17)
   [OK] Identity 2x2: Verified (Relative Residual: 0.00e+00)
   [OK] Singular 2x2: Verified (Relative Residual: 0.00e+00)


## 3. Ma trận Nghịch đảo (Matrix Inverse)

Một ma trận vuông $A$ có nghịch đảo $A^{-1}$ khi và chỉ khi $\det(A) \neq 0$. 
Trong bài test này, chúng ta đặc biệt kiểm tra **đường lỗi (Error Path)**: Nếu truyền vào ma trận suy biến, hàm phải ném ra lỗi `ValueError`.

In [4]:
print("--- Testing Inverse & Exception Handling ---")
inv_cases = [
    {"name": "Invertible 2x2", "A": [[4, 7], [2, 6]], "fail": False},
    {"name": "Singular (Expected Fail)", "A": [[1, 2], [2, 4]], "fail": True}
]

for case in inv_cases:
    if case["fail"]:
        try:
            inverse(case["A"])
            assert False, f"❌ {case['name']}: Lẽ ra phải báo lỗi ValueError nhưng không thấy!"
        except ValueError:
            print(f"   [OK] {case['name']}: Đã bắt được lỗi ValueError đúng như thiết kế.")
    else:
        A_inv = inverse(case["A"])
        # Kiểm tra A * A_inv = I
        res = np.dot(np.array(case["A"]), np.array(A_inv))
        assert np.allclose(res, np.eye(len(case["A"])), atol=1e-7)
        print(f"   [OK] {case['name']}: A * A^-1 khớp với Identity Matrix.")

--- Testing Inverse & Exception Handling ---
   [OK] Invertible 2x2: A * A^-1 khớp với Identity Matrix.
   [OK] Singular (Expected Fail): Đã bắt được lỗi ValueError đúng như thiết kế.


## 4. Phân tích Hệ số điều kiện (Ill-conditioned Analysis)

Ma trận điều kiện xấu là ma trận cực kỳ nhạy cảm với các thay đổi nhỏ ở đầu vào. 
Ví dụ: Một thay đổi $0.0001$ ở vector $b$ dẫn đến sai lệch khổng lồ ở vector $x$. 
Đây là bài học quan trọng về **Độ ổn định số học**.

In [5]:
print("--- Phân tích Hệ số điều kiện (Ill-conditioned Matrix) ---")

# Thiết lập ma trận A nhạy cảm
A_ill = [[1, 1], [1, 1.0001]]
b1 = [2, 2.0001]
b2 = [2, 2.0002] # Chỉ thay đổi 0.0001 so với b1

# Giải hai hệ phương trình
_, x1, _ = gaussian_eliminate(A_ill, b1)
_, x2, _ = gaussian_eliminate(A_ill, b2)

print(f"   [b1] -> Nghiệm x1: {x1}")
print(f"   [b2] -> Nghiệm x2: {x2}")

# Assertion: Chứng minh rằng nghiệm thay đổi rất lớn (sai lệch > 0.5) 
# dù đầu vào chỉ thay đổi cực nhỏ (0.0001)
diff = np.linalg.norm(np.array(x1) - np.array(x2))
assert diff > 0.5, f"❌ Lỗi: Hệ thống không thể hiện tính điều kiện xấu (Diff: {diff:.2f})"

print(f"✅ Minh chứng thành công: Độ lệch nghiệm là {diff:.2f} (Rất lớn so với độ lệch đầu vào 0.0001)")

--- Phân tích Hệ số điều kiện (Ill-conditioned Matrix) ---
   [b1] -> Nghiệm x1: [0.9999999999977796, 1.0000000000022204]
   [b2] -> Nghiệm x2: [0.0, 2.0]
✅ Minh chứng thành công: Độ lệch nghiệm là 1.41 (Rất lớn so với độ lệch đầu vào 0.0001)


## 5. Hạng và Không gian cơ sở (Rank & Basis)

Hàm `rank_and_basis` thực hiện biến đổi ma trận về dạng **RREF (Reduced Row Echelon Form)** để xác định:
1. **Rank (Hạng):** Số lượng các cột chứa phần tử trụ (pivot).
2. **Column Space:** Cơ sở được trích xuất từ các cột tương ứng trên ma trận gốc $A$.
3. **Null Space:** Tìm các vector nghiệm của hệ phương trình thuần nhất $Ax = 0$.

In [6]:
print("--- Testing Rank & Basis Suite ---")

rb_test_cases = [
    {
        "name": "Full Rank Square", 
        "A": [[1, 0, 0], [0, 1, 0], [0, 0, 1]], 
        "expected_rank": 3
    },
    {
        "name": "Rank Deficient (Dependent Rows)", 
        "A": [[1, 2, 3], [2, 4, 6], [3, 6, 9]], 
        "expected_rank": 1
    },
    {
        "name": "Rectangular Matrix", 
        "A": [[1, 2, 1, 3], [2, 4, 0, 4], [3, 6, 1, 7]], 
        "expected_rank": 2
    }
]

for case in rb_test_cases:
    rank, col_s, row_s, null_s = rank_and_basis(case["A"])
    
    # Assert hạng của ma trận phải khớp với dự kiến
    assert rank == case["expected_rank"], f"❌ {case['name']} FAILED: Rank expected {case['expected_rank']}, got {rank}"
    
    # Kiểm tra tính chất Null Space: A * null_vector phải xấp xỉ 0
    if len(null_s) > 0:
        for v in null_s:
            check_zero = np.dot(np.array(case["A"]), np.array(v))
            assert np.allclose(check_zero, 0, atol=1e-7), f"❌ {case['name']}: Null space vector is invalid!"
            
    print(f"   [OK] {case['name']}: Rank = {rank}, Nullity = {len(null_s)} (Verified)")

print("\n🚀 TẤT CẢ CÁC BÀI TEST ĐÃ VƯỢT QUA (ALL ASSERTS PASSED)")

--- Testing Rank & Basis Suite ---
   [OK] Full Rank Square: Rank = 3, Nullity = 0 (Verified)
   [OK] Rank Deficient (Dependent Rows): Rank = 1, Nullity = 2 (Verified)
   [OK] Rectangular Matrix: Rank = 2, Nullity = 2 (Verified)

🚀 TẤT CẢ CÁC BÀI TEST ĐÃ VƯỢT QUA (ALL ASSERTS PASSED)
